# تحليل الطلب الأسبوعي على الوجبات

دفتر قابل لإعادة التشغيل يغطي فحص البيانات، اتجاهات الطلب، الفئات والمراكز، اختبارات الأسعار والعروض، والرسوم. يمكن تشغيل خلية التنبؤ الاختيارية في النهاية بعد اكتمال التحليل.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'food_demand_analysis.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
import food_demand_analysis as project

train, test, quality = project.load_and_validate(ROOT / 'data' / 'raw')
train['revenue'] = train['num_orders'] * train['checkout_price']
print(quality)

## 1. فحص سلامة البيانات

لا تُحذف السجلات أو تُعوّض القيم بصمت. يفشل التحقق برسالة واضحة عند اختلال المفاتيح أو المراجع.

In [ ]:
display(train.describe(include='all').T)
display(train[['week', 'center_id', 'meal_id', 'num_orders']].head())

## 2. الطلب الأسبوعي والفئات والمطابخ

تُقاس المساهمة بإجمالي الطلبات؛ وتُعرض كذلك قيمة الطلبات المقدرة بوصفها مؤشراً للإيرادات لا للربح.

In [ ]:
weekly = (train.groupby('week', as_index=False)
          .agg(total_orders=('num_orders', 'sum'), average_orders=('num_orders', 'mean'),
               revenue=('revenue', 'sum')))
category = project.summarize_group(train, 'category', 'category')
cuisine = project.summarize_group(train, 'cuisine', 'cuisine')
display(weekly.head())
display(category)
display(cuisine)

## 3. مساهمة المراكز والوجبات

يُستخرج أقصر ترتيب تراكمي يغطي 80% من الطلبات، بدلاً من افتراض نسبة ثابتة من المراكز أو الوجبات.

In [ ]:
centers = project.summarize_group(train, 'center_id', 'center_id')
meals = project.summarize_group(train, 'meal_id', 'meal_id')
top_categories = project.top_contributors(category)
top_centers = project.top_contributors(centers)
display(top_categories)
display(top_centers)
display(meals.head(10))

## 4. اختبارات الأسعار والعروض

يقارن Mann–Whitney توزيعات الطلب بين السجلات المعروضة وغير المعروضة، ويقيس Spearman الارتباط الرتبي بين نسبة الخصم والطلب. لا تُفسر هذه المقارنات الرصدية على أنها أثر سببي.

In [ ]:
promotion_tests, statistics = project._promotion_tests(train)
display(promotion_tests)
print(statistics)

## 5. الرسوم البيانية

يولد المسار أربع صور ثابتة في `reports/figures/` لإدراجها في التقرير والعرض.

In [ ]:
project.run_analysis(ROOT / 'data' / 'raw')
for filename in ['weekly_orders.png', 'orders_by_category.png', 'orders_by_center.png', 'discount_and_orders.png']:
    display(Image(filename=str(ROOT / 'reports' / 'figures' / filename)))

## 6. التنبؤ الاختياري

يُستخدم الأسبوعان 136–145 للتحقق الزمني، وتُقارن الغابة العشوائية بالانحدار الخطي وفق MAE وRMSE. بعد اختيار النموذج يعاد تدريبه على كامل بيانات التدريب لإنتاج تنبؤات الأسابيع 146–155.

In [ ]:
model_results = project.train_and_predict(train, test, ROOT / 'reports')
display(model_results)
print('Prediction file:', ROOT / 'reports' / 'submission.csv')